# SMART Checkpoint Simulation/Evaluation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-constrained/notebooks/smart_checkpoint_eval_colab.ipynb)

## Objective
- Load trained checkpoints from Drive and run eval-only SMART validation commands.
- Persist model-eval plans, ingested metrics, and reproducibility artifacts.
- Do not retrain here; this notebook is strictly for simulation/evaluation.


In [ ]:
# Step 1: Sync repo + deterministic Colab bootstrap
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for path in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load smart-constrained config + run context
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "smart-constrained"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))
CONSTRAINTS = dict(cfg.get("constraints", {}))
EVAL = dict(cfg.get("evaluation", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_constrained"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)

print("RUN_TAG:", RUN_TAG)
print("persist_run_dir:", persist_run_dir)
print("cfg_hash:", cfg_hash)


In [ ]:
# Step 3: Define checkpoint models to evaluate
# Required env vars (recommended):
#   SMART_BASELINE_CKPT=/content/drive/MyDrive/.../baseline.ckpt
#   SMART_VARIANT_CKPTS=/content/drive/.../variant1.ckpt,/content/drive/.../variant2.ckpt
# Optional:
#   WOSAC_MODEL_METRICS_DIR=/content/drive/.../model_metrics_json

baseline_ckpt = os.environ.get("SMART_BASELINE_CKPT", "").strip()
variant_ckpts = [v.strip() for v in os.environ.get("SMART_VARIANT_CKPTS", "").split(",") if v.strip()]
metrics_dir = os.environ.get("WOSAC_MODEL_METRICS_DIR", "").strip()

assert baseline_ckpt, "Set SMART_BASELINE_CKPT before running eval flow"

models = [{"model_id": "smart_baseline", "checkpoint_path": baseline_ckpt, "env": {}}]
for idx, ckpt in enumerate(variant_ckpts, start=1):
    models.append({"model_id": f"variant_{idx}", "checkpoint_path": ckpt, "env": {}})

print("num_models:", len(models))
print("metrics_dir:", metrics_dir or "<not set>")


In [ ]:
# Step 4: Build checkpoint eval commands and ingest any available metrics
from src.workflows import run_smart_eval_flow

flow_bundle = run_smart_eval_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    smart_repo_url=str(SMART.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART.get("branch", "main")),
    smart_repo_dir=str(SMART.get("repo_dir", "/content/SMART")),
    smart_val_config=str(SMART.get("val_config", "configs/validation/validation_scalable.yaml")),
    sync_smart_repo=True,
    models=models,
    metrics_dir=metrics_dir,
)

print("summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
print("artifacts:")
print(json.dumps(flow_bundle.artifacts, indent=2, sort_keys=True))
print("first_model:")
print(json.dumps(flow_bundle.models[0], indent=2, sort_keys=True))


In [ ]:
# Step 5: Optional validate-command execution (eval-only; no training)
RUN_VALIDATE_ALL = False
MODEL_IDS_TO_RUN = []  # e.g. ["smart_baseline", "variant_1"]

if RUN_VALIDATE_ALL:
    selected = list(flow_bundle.models)
elif MODEL_IDS_TO_RUN:
    selected = [m for m in flow_bundle.models if m["model_id"] in set(MODEL_IDS_TO_RUN)]
else:
    selected = []

for idx, model in enumerate(selected, start=1):
    cmd = model["validate_cmd"]
    print(f"[eval {idx}/{len(selected)}] {model['model_id']} -> {cmd}")
    subprocess.run(["bash", "-lc", cmd], check=True)

print("Checkpoint-eval execution block done.")


In [ ]:
# Step 6: Write checkpoint eval artifact (repo + Drive)
payload = {
    "run_id": "smart_checkpoint_eval_v0",
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_tag": RUN_TAG,
    "primary_metric": str(EVAL.get("primary_metric", "realism_meta_metric")),
    "models": flow_bundle.models,
    "summary": flow_bundle.summary,
    "artifacts": flow_bundle.artifacts,
    "constraints": CONSTRAINTS,
    "provenance": {
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

repo_path = Path("experiments/smart-constrained/results/smart_checkpoint_eval_v0.json")
repo_path.parent.mkdir(parents=True, exist_ok=True)
repo_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

drive_path = persist_run_dir / "outputs" / "smart_checkpoint_eval_v0.json"
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("repo_path:", repo_path)
print("drive_path:", drive_path)
